In [ ]:
from google.colab import files
uploaded = files.upload()

Saving nba_2024_25_player_per_game.csv to nba_2024_25_player_per_game.csv
Saving nba_2024-25_player_shooting.csv to nba_2024-25_player_shooting.csv
Saving nba_2024-25_player_per100pos.csv to nba_2024-25_player_per100pos.csv
Saving nba_2024-25_advanced.csv to nba_2024-25_advanced.csv


In [ ]:
!pip install nba_stats_tracking

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.1/154.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.8 MB/s eta 0:00:00
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.11.4
    Uninstalling pydantic-2.11.4:
      Successfully uninstalled pydantic-2.11.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 0.3.60 requires pydantic>=2.7.4, but you have pydantic 1.10.22 which is incompatible.
google-genai 1.16.1 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.22 which is incompatible.
thinc 8.3.6 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.22 which is incompatible.
langchain 0.3.25 requires pydantic<3.0.0,>=2.7.4, but you have pydantic 1.10.22 which is incompatible.
albumentations 2.0.7 requires pydantic>=2.9.2, but you have pydantic 1.10.22 which is incompatib

In [ ]:
import pandas as pd
import re
import requests
from nba_stats_tracking import tracking
import time

In [ ]:
df = pd.read_csv('nba_2020-21_player_per100pos.csv')
df2 = pd.read_csv('nba_2020-21_advanced.csv')
df3 = pd.read_csv('nba_2020-21_player_shooting.csv', header = [0, 1])

In [ ]:
df

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,TRB,AST,STL,BLK,TOV,PF,PTS,ORtg,DRtg,Awards
0,1.0,Julius Randle,26.0,NYK,PF,71.0,71.0,2667.0,11.3,24.8,...,13.6,8.0,1.2,0.3,4.5,4.2,32.1,111.0,107.0,"MVP-8,MIP-1,AS,NBA2"
1,2.0,RJ Barrett,20.0,NYK,SG,72.0,72.0,2511.0,9.3,21.1,...,8.2,4.3,1.1,0.4,2.8,3.7,25.2,106.0,110.0,NaN
2,3.0,Nikola Jokić,25.0,DEN,C,72.0,72.0,2488.0,14.5,25.7,...,15.5,11.9,1.9,1.0,4.4,3.8,37.7,130.0,109.0,"MVP-1,AS,NBA1"
3,4.0,Buddy Hield,28.0,SAC,SG,71.0,71.0,2433.0,8.0,19.6,...,6.6,5.1,1.2,0.6,2.6,3.5,23.2,109.0,119.0,NaN
4,5.0,Damian Lillard,30.0,POR,PG,67.0,67.0,2398.0,12.2,27.1,...,5.8,10.3,1.3,0.3,4.1,2.1,39.2,125.0,118.0,"MVP-7,AS,NBA2"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
701,537.0,Anžejs Pasečņiks,25.0,WAS,C,1.0,0.0,6.0,0.0,7.7,...,7.7,7.7,0.0,0.0,38.4,15.4,0.0,21.0,120.0,NaN
702,538.0,Ashton Hagans,21.0,MIN,PG,2.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,11.8,0.0,0.0,0.0,124.0,NaN
703,539.0,Udonis Haslem,40.0,MIA,C,1.0,0.0,3.0,33.1,33.1,...,16.6,0.0,0.0,0.0,0.0,0.0,66.2,200.0,111.0,NaN
704,540.0,Will Magnay,22.0,NOP,C,1.0,0.0,3.0,0.0,16.0,...,0.0,0.0,0.0,0.0,16.0,16.0,0.0,0.0,122.0,NaN


In [ ]:
df.columns

Index(['Rk', 'Player', 'Age', 'Team', 'Pos', 'G', 'GS', 'MP', 'FG', 'FGA',
       'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA',
       'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS',
       'ORtg', 'DRtg', 'Awards'],
      dtype='object')

In [ ]:
df2

,Rk,Player,Age,Team,Pos,G,GS,MP,PER,TS%,...,USG%,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP,Awards
0,1.0,Julius Randle,26.0,NYK,PF,71.0,71.0,2667.0,19.7,0.567,...,29.3,3.4,4.3,7.8,0.140,2.9,0.9,3.8,3.9,"MVP-8,MIP-1,AS,NBA2"
1,2.0,RJ Barrett,20.0,NYK,SG,72.0,72.0,2511.0,13.4,0.535,...,23.4,0.9,3.1,4.1,0.078,-0.9,-0.3,-1.2,0.5,NaN
2,3.0,Nikola Jokić,25.0,DEN,C,72.0,72.0,2488.0,31.3,0.647,...,29.6,12.2,3.4,15.6,0.301,9.1,3.0,12.1,8.8,"MVP-1,AS,NBA1"
3,4.0,Buddy Hield,28.0,SAC,SG,71.0,71.0,2433.0,12.8,0.567,...,20.7,1.5,0.7,2.2,0.044,1.0,-1.6,-0.6,0.9,NaN
4,5.0,Damian Lillard,30.0,POR,PG,67.0,67.0,2398.0,25.6,0.623,...,31.4,9.6,0.8,10.4,0.209,7.5,-1.3,6.3,5.0,"MVP-7,AS,NBA2"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
701,537.0,Anžejs Pasečņiks,25.0,WAS,C,1.0,0.0,6.0,-40.6,0.000,...,41.4,-0.1,0.0,-0.1,-1.113,-40.7,-5.9,-46.6,-0.1,NaN
702,538.0,Ashton Hagans,21.0,MIN,PG,2.0,0.0,4.0,-12.4,NaN,...,10.5,0.0,0.0,0.0,-0.353,-13.7,-7.4,-21.1,0.0,NaN
703,539.0,Udonis Haslem,40.0,MIA,C,1.0,0.0,3.0,54.6,1.000,...,30.1,0.0,0.0,0.0,0.475,24.1,7.0,31.1,0.0,NaN
704,540.0,Will Magnay,22.0,NOP,C,1.0,0.0,3.0,-35.1,0.000,...,28.0,0.0,0.0,0.0,-0.787,-30.7,-8.6,-39.3,0.0,NaN


In [ ]:
df2.columns

Index(['Rk', 'Player', 'Age', 'Team', 'Pos', 'G', 'GS', 'MP', 'PER', 'TS%',
       '3PAr', 'FTr', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%',
       'USG%', 'OWS', 'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP',
       'Awards'],
      dtype='object')

In [ ]:
df3

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
                    Rk             Player                Age   
0                  1.0      Julius Randle               26.0   
1                  2.0         RJ Barrett               20.0   
2                  3.0       Nikola Jokić               25.0   
3                  4.0        Buddy Hield               28.0   
4                  5.0     Damian Lillard               30.0   
..                 ...                ...                ...   
701              537.0   Anžejs Pasečņiks               25.0   
702              538.0      Ashton Hagans               21.0   
703              539.0      Udonis Haslem               40.0   
704              540.0        Will Magnay               22.0   
705                NaN     League Average                NaN   

    Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0  \
                  Team                Pos                  G   
0                  NYK                 PF               71.0   
1                  NYK                 SG               72.0   
2                  DEN                  C               72.0   
3                  SAC                 SG               71.0   
4                  POR                 PG               67.0   
..                 ...                ...                ...   
701                WAS                  C                1.0   
702                MIN                 PG                2.0   
703                MIA                  C                1.0   
704                NOP                  C                1.0   
705                NaN                NaN                NaN   

    Unnamed: 6_level_0 Unnamed: 7_level_0 Unnamed: 8_level_0  \
                    GS                 MP                FG%   
0                 71.0             2667.0              0.456   
1                 72.0             2511.0              0.441   
2                 72.0             2488.0              0.566   
3                 71.0             2433.0              0.406   
4                 67.0             2398.0              0.451   
..                 ...                ...                ...   
701                0.0                6.0              0.000   
702                0.0                4.0                NaN   
703                0.0                3.0              1.000   
704                0.0                3.0              0.000   
705                NaN                NaN              0.466   

    Unnamed: 9_level_0  ... FG% by Distance % of FG Ast'd         Dunks        \
                 Dist.  ...              3P            2P     3P   %FGA     #   
0                 14.5  ...           0.411         0.342  0.794  0.022  25.0   
1                 11.5  ...           0.401         0.353  0.976  0.040  39.0   
2                 10.6  ...           0.388         0.484  0.967  0.037  45.0   
3                 21.2  ...           0.391         0.331  0.812  0.009   8.0   
4                 17.9  ...           0.391         0.177  0.393  0.015  19.0   
..                 ...  ...             ...           ...    ...    ...   ...   
701               25.2  ...           0.000           NaN    NaN  0.000   0.0   
702                NaN  ...             NaN           NaN    NaN    NaN   0.0   
703                8.6  ...             NaN         1.000    NaN  0.000   0.0   
704               26.7  ...           0.000           NaN    NaN  0.000   0.0   
705               13.7  ...           0.367         0.502  0.826  0.054   NaN   

    Corner 3s        Heaves       Unnamed: 30_level_0  
         %3PA    3P%   Att.  Md.               Awards  
0       0.219  0.412    1.0  0.0  MVP-8,MIP-1,AS,NBA2  
1       0.427  0.424    1.0  0.0                  NaN  
2       0.034  0.375    3.0  0.0        MVP-1,AS,NBA1  
3       0.136  0.469    1.0  0.0                  NaN  
4       0.038  0.185    4.0  0.0        MVP-7,AS,NBA2  
..        ...    ...    ...  ...                  ...  
701

In [ ]:
df3.columns

MultiIndex([(  'Unnamed: 0_level_0',     'Rk'),
            (  'Unnamed: 1_level_0', 'Player'),
            (  'Unnamed: 2_level_0',    'Age'),
            (  'Unnamed: 3_level_0',   'Team'),
            (  'Unnamed: 4_level_0',    'Pos'),
            (  'Unnamed: 5_level_0',      'G'),
            (  'Unnamed: 6_level_0',     'GS'),
            (  'Unnamed: 7_level_0',     'MP'),
            (  'Unnamed: 8_level_0',    'FG%'),
            (  'Unnamed: 9_level_0',  'Dist.'),
            ('% of FGA by Distance',     '2P'),
            ('% of FGA by Distance',    '0-3'),
            ('% of FGA by Distance',   '3-10'),
            ('% of FGA by Distance',  '10-16'),
            ('% of FGA by Distance',  '16-3P'),
            ('% of FGA by Distance',     '3P'),
            (     'FG% by Distance',     '2P'),
            (     'FG% by Distance',    '0-3'),
            (     'FG% by Distance',   '3-10'),
            (     'FG% by Distance',  '10-16'),
            (     'FG% by Distance',  '1

In [ ]:
#Creating functions to perform simple, preliminary cleaning
def reformat_standard_stats(dataframe):
    """Input 1 dataframe cau thu tu web Basketball Reference(per game/per 100 possessions)
    va loai bo cac cot khong can thiet cho data analysis hoac modeling."""
    df_new = dataframe.drop(columns=['Rk', 'Awards'])

    return df_new

def reformat_adv_stats(dataframe):
    """Input dataframe tu advanced player stats tu web Basketball Reference va loai bo cac cot
    khong can thiet cho data analysis hoac modeling. Cung nhu loai bo cac cot trung voi
    cac cot trong standard statistics dataframe."""
    df_new = dataframe.drop(columns=['Rk', 'Player', 'Age', 'Team', 'Pos', 'G', 'GS', 'MP'])

    return df_new


In [ ]:
def join_dataframes(stand_df, adv_df, shooting_stats_df):
    """Joins standard and advanced dataframes from Basketball Reference into a singular dataframes.  Appends
    selected columns from the shooting statistics dataframe.

    stand_df: A dataframe with standard player statistics from Basketball Reference. For example see this
    website: https://www.basketball-reference.com/leagues/NBA_2018_per_poss.html

    adv_df: A dataframe with advanced player statistics from Basketball Reference. For example see this
    website: https://www.basketball-reference.com/leagues/NBA_2018_advanced.html

    shooting_stats_df: A dataframe with player shooting statistics from Basketball Reference. For example
    see this website: https://www.basketball-reference.com/leagues/NBA_2018_shooting.html

    year_range: The year range for an NBA season."""

    df_std_reformat = reformat_standard_stats(stand_df)
    df_adv_reformat = reformat_adv_stats(adv_df)

    df_joined_two = df_std_reformat.join(df_adv_reformat) #join standard and advanced player stats dataframes

    #creating new columns with data from shooting statistics dataframe
    df_joined_two['pct_of_FGA_twos'] = shooting_stats_df['% of FGA by Distance']['2P']
    df_joined_two['pct_of_FGA_threes'] = shooting_stats_df['% of FGA by Distance']['3P']
    df_joined_two['two_pt_fg_pct'] = shooting_stats_df['FG% by Distance']['2P']
    df_joined_two['three_pt_fg_pct'] = shooting_stats_df['FG% by Distance']['3P']

    df_final = df_joined_two.drop(columns=['Awards'])
    #loai bo dong cuoi khong chua gia tri cau thu
    df_final = df_final.iloc[:-1]
    return df_final

In [ ]:
#check to make sure dataframes from the same year are the same length before running functions
len(df) == len(df2) == len(df3)

True

In [ ]:
bballref_2021_df = join_dataframes(df, df2, df3)
bballref_2021_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 705 entries, 0 to 704
Data columns (total 55 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Player             705 non-null    object 
 1   Age                705 non-null    float64
 2   Team               705 non-null    object 
 3   Pos                705 non-null    object 
 4   G                  705 non-null    float64
 5   GS                 705 non-null    float64
 6   MP                 705 non-null    float64
 7   FG                 705 non-null    float64
 8   FGA                705 non-null    float64
 9   FG%                703 non-null    float64
 10  3P                 705 non-null    float64
 11  3PA                705 non-null    float64
 12  3P%                670 non-null    float64
 13  2P                 705 non-null    float64
 14  2PA                705 non-null    float64
 15  2P%                699 non-null    float64
 16  eFG%               703 non

In [ ]:
bballref_2021_df

,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,...,WS,WS/48,OBPM,DBPM,BPM,VORP,pct_of_FGA_twos,pct_of_FGA_threes,two_pt_fg_pct,three_pt_fg_pct
0,Julius Randle,26.0,NYK,PF,71.0,71.0,2667.0,11.3,24.8,0.456,...,7.8,0.140,2.9,0.9,3.8,3.9,0.706,0.294,0.474,0.411
1,RJ Barrett,20.0,NYK,SG,72.0,72.0,2511.0,9.3,21.1,0.441,...,4.1,0.078,-0.9,-0.3,-1.2,0.5,0.708,0.292,0.457,0.401
2,Nikola Jokić,25.0,DEN,C,72.0,72.0,2488.0,14.5,25.7,0.566,...,15.6,0.301,9.1,3.0,12.1,8.8,0.817,0.183,0.606,0.388
3,Buddy Hield,28.0,SAC,SG,71.0,71.0,2433.0,8.0,19.6,0.406,...,2.2,0.044,1.0,-1.6,-0.6,0.9,0.273,0.727,0.446,0.391
4,Damian Lillard,30.0,POR,PG,67.0,67.0,2398.0,12.2,27.1,0.451,...,10.4,0.209,7.5,-1.3,6.3,5.0,0.472,0.528,0.519,0.391
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
700,Iman Shumpert,30.0,BRK,SG,2.0,0.0,11.0,4.4,17.5,0.250,...,0.0,-0.212,-10.4,-2.2,-12.6,0.0,0.250,0.750,1.000,0.000
701,Anžejs Pasečņiks,25.0,WAS,C,1.0,0.0,6.0,0.0,7.7,0.000,...,-0.1,-1.113,-40.7,-5.9,-46.6,-0.1,0.000,1.000,NaN,0.000
702,Ashton Hagans,21.0,MIN,PG,2.0,0.0,4.0,0.0,0.0,NaN,...,0.0,-0.353,-13.7,-7.4,-21.1,0.0,NaN,NaN,NaN,NaN
703,Udonis Haslem,40.0,MIA,C,1.0,0.0,3.0,33.1,33.1,1.000,...,0.0,0.475,24.1,7.0,31.1,0.0,1.000,0.000,1.000,NaN


Gathering Tracking Data from stats.nba.com via nba_tracking_stats api
Select drives, CatchShoot, Passing, Possessions, SpeedDistance, PullUpShot

In [ ]:
#preview statistics related to drives to the basket

stat_measure = 'Drives'
seasons = ['2020-21']
season_types = ['Regular Season']
entity_type = 'Player'

stats, league_totals = tracking.aggregate_full_season_tracking_stats_for_seasons(
    stat_measure,
    seasons,
    season_types,
    entity_type)



pd.DataFrame(stats)

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,"(player_id, 203932)","(player_name, Aaron Gordon)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612743)","(team_abbreviation, DEN)","(games_played, 50)","(wins, 29)",...,"(drives, 228.0)","(fgm, 39.0)","(fga, 90.0)","(ftm, 24.0)","(fta, 39.0)","(points, 102.0)","(passes, 83.0)","(assists, 19.0)","(turnovers, 19.0)","(fouls, 20.0)"
1,"(player_id, 1628988)","(player_name, Aaron Holiday)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612754)","(team_abbreviation, IND)","(games_played, 66)","(wins, 30)",...,"(drives, 383.0)","(fgm, 60.0)","(fga, 167.0)","(ftm, 26.0)","(fta, 32.0)","(points, 146.0)","(passes, 160.0)","(assists, 39.0)","(turnovers, 29.0)","(fouls, 16.0)"
2,"(player_id, 1630174)","(player_name, Aaron Nesmith)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612738)","(team_abbreviation, BOS)","(games_played, 46)","(wins, 22)",...,"(drives, 73.0)","(fgm, 12.0)","(fga, 28.0)","(ftm, 8.0)","(fta, 12.0)","(points, 33.0)","(passes, 30.0)","(assists, 7.0)","(turnovers, 6.0)","(fouls, 6.0)"
3,"(player_id, 1627846)","(player_name, Abdel Nader)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612756)","(team_abbreviation, PHX)","(games_played, 24)","(wins, 16)",...,"(drives, 83.0)","(fgm, 20.0)","(fga, 38.0)","(ftm, 7.0)","(fta, 12.0)","(points, 48.0)","(passes, 26.0)","(assists, 8.0)","(turnovers, 8.0)","(fouls, 6.0)"
4,"(player_id, 1629690)","(player_name, Adam Mokoka)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612741)","(team_abbreviation, CHI)","(games_played, 14)","(wins, 3)",...,"(drives, 5.0)","(fgm, 1.0)","(fga, 3.0)","(ftm, 0.0)","(fta, 0.0)","(points, 2.0)","(passes, 1.0)","(assists, 1.0)","(turnovers, 0.0)","(fouls, 0.0)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,"(player_id, 1627812)","(player_name, Yogi Ferrell)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612746)","(team_abbreviation, LAC)","(games_played, 10)","(wins, 4)",...,"(drives, 28.0)","(fgm, 4.0)","(fga, 11.0)","(ftm, 2.0)","(fta, 2.0)","(points, 10.0)","(passes, 13.0)","(assists, 5.0)","(turnovers, 1.0)","(fouls, 1.0)"
536,"(player_id, 1629139)","(player_name, Yuta Watanabe)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612761)","(team_abbreviation, TOR)","(games_played, 50)","(wins, 24)",...,"(drives, 78.0)","(fgm, 10.0)","(fga, 29.0)","(ftm, 10.0)","(fta, 12.0)","(points, 32.0)","(passes, 38.0)","(assists, 10.0)","(turnovers, 3.0)","(fouls, 6.0)"
537,"(player_id, 203897)","(player_name, Zach LaVine)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612741)","(team_abbreviation, CHI)","(games_played, 58)","(wins, 26)",...,"(drives, 697.0)","(fgm, 184.0)","(fga, 345.0)","(ftm, 101.0)","(fta, 119.0)","(points, 485.0)","(passes, 192.0)","(assists, 51.0)","(turnovers, 66.0)","(fouls, 60.0)"
538,"(player_id, 1630192)","(player_name, Zeke Nnaji)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612743)","(team_abbreviation, DEN)","(games_played, 42)","(wins, 30)",...,"(drives, 8.0)","(fgm, 0.0)","(fga, 4.0)","(ftm, 1.0)","(fta, 2.0)","(points, 1.0)","(passes, 2.0)","(assists, 0.0)","(turnovers, 1.0)","(fouls, 1.0)"


In [ ]:
#preview statistics related to catch and shoot attempts
stat_measure = 'CatchShoot'
seasons = ['2020-21']
season_types = ['Regular Season']
entity_type = 'Player'
stats, league_totals = tracking.aggregate_full_season_tracking_stats_for_seasons(
    stat_measure,
    seasons,
    season_types,
    entity_type
)
pd.DataFrame(stats)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,"(player_id, 203932)","(player_name, Aaron Gordon)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612743)","(team_abbreviation, DEN)","(games_played, 50)","(wins, 29)","(losses, 21)","(minutes, 1384.0)","(fgm, 48.0)","(fga, 123.0)","(points, 137.0)","(fg3m, 41.0)","(fg3a, 109.0)"
1,"(player_id, 1628988)","(player_name, Aaron Holiday)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612754)","(team_abbreviation, IND)","(games_played, 66)","(wins, 30)","(losses, 36)","(minutes, 1176.0)","(fgm, 53.0)","(fga, 137.0)","(points, 159.0)","(fg3m, 53.0)","(fg3a, 135.0)"
2,"(player_id, 1630174)","(player_name, Aaron Nesmith)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612738)","(team_abbreviation, BOS)","(games_played, 46)","(wins, 22)","(losses, 24)","(minutes, 669.0)","(fgm, 37.0)","(fga, 93.0)","(points, 111.0)","(fg3m, 37.0)","(fg3a, 93.0)"
3,"(player_id, 1627846)","(player_name, Abdel Nader)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612756)","(team_abbreviation, PHX)","(games_played, 24)","(wins, 16)","(losses, 8)","(minutes, 355.0)","(fgm, 18.0)","(fga, 42.0)","(points, 54.0)","(fg3m, 18.0)","(fg3a, 41.0)"
4,"(player_id, 1629690)","(player_name, Adam Mokoka)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612741)","(team_abbreviation, CHI)","(games_played, 14)","(wins, 3)","(losses, 11)","(minutes, 56.0)","(fgm, 1.0)","(fga, 8.0)","(points, 3.0)","(fg3m, 1.0)","(fg3a, 8.0)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,"(player_id, 1627812)","(player_name, Yogi Ferrell)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612746)","(team_abbreviation, LAC)","(games_played, 10)","(wins, 4)","(losses, 6)","(minutes, 137.0)","(fgm, 7.0)","(fga, 14.0)","(points, 20.0)","(fg3m, 6.0)","(fg3a, 11.0)"
536,"(player_id, 1629139)","(player_name, Yuta Watanabe)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612761)","(team_abbreviation, TOR)","(games_played, 50)","(wins, 24)","(losses, 26)","(minutes, 723.0)","(fgm, 36.0)","(fga, 87.0)","(points, 106.0)","(fg3m, 34.0)","(fg3a, 85.0)"
537,"(player_id, 203897)","(player_name, Zach LaVine)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612741)","(team_abbreviation, CHI)","(games_played, 58)","(wins, 26)","(losses, 32)","(minutes, 2034.0)","(fgm, 77.0)","(fga, 165.0)","(points, 226.0)","(fg3m, 72.0)","(fg3a, 148.0)"
538,"(player_id, 1630192)","(player_name, Zeke Nnaji)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612743)","(team_abbreviation, DEN)","(games_played, 42)","(wins, 30)","(losses, 12)","(minutes, 397.0)","(fgm, 26.0)","(fga, 64.0)","(points, 75.0)","(fg3m, 23.0)","(fg3a, 58.0)"


In [ ]:
#preview statistics related to drives to how often a player has the ball
stat_measure = 'Possessions'
seasons = ['2020-21']
season_types = ['Regular Season']
entity_type = 'Player'
stats, league_totals = tracking.aggregate_full_season_tracking_stats_for_seasons(
    stat_measure,
    seasons,
    season_types,
    entity_type
)
pd.DataFrame(stats)

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,"(player_id, 203932)","(player_name, Aaron Gordon)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612743)","(team_abbreviation, DEN)","(games_played, 50)","(wins, 29)",...,"(minutes, 1384.0)","(points, 618.0)","(touches, 2415.0)","(front_court_touches, 1291.0)","(time_of_poss, 145.8)","(elbow_touches, 63.0)","(post_touches, 113.0)","(paint_touches, 192.0)","(seconds_per_touch, 3.62)","(dribbles_per_touch, 2.83)"
1,"(player_id, 1628988)","(player_name, Aaron Holiday)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612754)","(team_abbreviation, IND)","(games_played, 66)","(wins, 30)",...,"(minutes, 1176.0)","(points, 475.0)","(touches, 1620.0)","(front_court_touches, 1020.0)","(time_of_poss, 80.1)","(elbow_touches, 11.0)","(post_touches, 0.0)","(paint_touches, 22.0)","(seconds_per_touch, 2.97)","(dribbles_per_touch, 2.61)"
2,"(player_id, 1630174)","(player_name, Aaron Nesmith)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612738)","(team_abbreviation, BOS)","(games_played, 46)","(wins, 22)",...,"(minutes, 669.0)","(points, 218.0)","(touches, 767.0)","(front_court_touches, 435.0)","(time_of_poss, 21.4)","(elbow_touches, 8.0)","(post_touches, 1.0)","(paint_touches, 33.0)","(seconds_per_touch, 1.67)","(dribbles_per_touch, 0.81)"
3,"(player_id, 1627846)","(player_name, Abdel Nader)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612756)","(team_abbreviation, PHX)","(games_played, 24)","(wins, 16)",...,"(minutes, 355.0)","(points, 160.0)","(touches, 389.0)","(front_court_touches, 261.0)","(time_of_poss, 12.4)","(elbow_touches, 0.0)","(post_touches, 0.0)","(paint_touches, 17.0)","(seconds_per_touch, 1.91)","(dribbles_per_touch, 1.19)"
4,"(player_id, 1629690)","(player_name, Adam Mokoka)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612741)","(team_abbreviation, CHI)","(games_played, 14)","(wins, 3)",...,"(minutes, 56.0)","(points, 15.0)","(touches, 58.0)","(front_court_touches, 43.0)","(time_of_poss, 1.3)","(elbow_touches, 1.0)","(post_touches, 0.0)","(paint_touches, 1.0)","(seconds_per_touch, 1.37)","(dribbles_per_touch, 0.78)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,"(player_id, 1627812)","(player_name, Yogi Ferrell)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612746)","(team_abbreviation, LAC)","(games_played, 10)","(wins, 4)",...,"(minutes, 137.0)","(points, 56.0)","(touches, 246.0)","(front_court_touches, 91.0)","(time_of_poss, 22.3)","(elbow_touches, 4.0)","(post_touches, 0.0)","(paint_touches, 8.0)","(seconds_per_touch, 5.44)","(dribbles_per_touch, 4.78)"
536,"(player_id, 1629139)","(player_name, Yuta Watanabe)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612761)","(team_abbreviation, TOR)","(games_played, 50)","(wins, 24)",...,"(minutes, 723.0)","(points, 218.0)","(touches, 953.0)","(front_court_touches, 541.0)","(time_of_poss, 26.5)","(elbow_touches, 23.0)","(post_touches, 1.0)","(paint_touches, 51.0)","(seconds_per_touch, 1.67)","(dribbles_per_touch, 0.98)"
537,"(player_id, 203897)","(player_name, Zach LaVine)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612741)","(team_abbreviation, CHI)","(games_played, 58)","(wins, 26)",...,"(minutes, 2034.0)","(points, 1591.0)","(touches, 3946.0)","(front_court_touches, 1966.0)","(time_of_poss, 314.9)","(elbow_touches, 50.0)","(post_touches, 27.0)","(paint_touches, 77.0)","(seconds_per_touch, 4.79)","(dribbles_per_touch, 4.33)"
538,

In [ ]:
# (TEST) Convert the list of tuples to a DataFrame
dataframes = []
for entry in stats:
    data_dict = dict(entry)  # Convert each entry to a dictionary
    dataframes.append(data_dict)

# Create a DataFrame from the list of dictionaries
df_test = pd.DataFrame(dataframes)

# Display the DataFrame
df_test.head()

,player_id,player_name,team_name,season,game_id,opponent_team_id,team_id,team_abbreviation,games_played,wins,...,minutes,points,touches,front_court_touches,time_of_poss,elbow_touches,post_touches,paint_touches,seconds_per_touch,dribbles_per_touch
0,203932,Aaron Gordon,None,2020-21 Regular Season,None,None,1610612743,DEN,50,29,...,1384.0,618.0,2415.0,1291.0,145.8,63.0,113.0,192.0,3.62,2.83
1,1628988,Aaron Holiday,None,2020-21 Regular Season,None,None,1610612754,IND,66,30,...,1176.0,475.0,1620.0,1020.0,80.1,11.0,0.0,22.0,2.97,2.61
2,1630174,Aaron Nesmith,None,2020-21 Regular Season,None,None,1610612738,BOS,46,22,...,669.0,218.0,767.0,435.0,21.4,8.0,1.0,33.0,1.67,0.81
3,1627846,Abdel Nader,None,2020-21 Regular Season,None,None,1610612756,PHX,24,16,...,355.0,160.0,389.0,261.0,12.4,0.0,0.0,17.0,1.91,1.19
4,1629690,Adam Mokoka,None,2020-21 Regular Season,None,None,1610612741,CHI,14,3,...,56.0,15.0,58.0,43.0,1.3,1.0,0.0,1.0,1.37,0.78


In [ ]:
df_test.columns

Index(['player_id', 'player_name', 'team_name', 'season', 'game_id',
       'opponent_team_id', 'team_id', 'team_abbreviation', 'games_played',
       'wins', 'losses', 'minutes', 'points', 'touches', 'front_court_touches',
       'time_of_poss', 'elbow_touches', 'post_touches', 'paint_touches',
       'seconds_per_touch', 'dribbles_per_touch'],
      dtype='object')

In [ ]:
#preview statistics related to pull up shots
stat_measure = 'PullUpShot'
seasons = ['2020-21']
season_types = ['Regular Season']
entity_type = 'Player'
stats, league_totals = tracking.aggregate_full_season_tracking_stats_for_seasons(
    stat_measure,
    seasons,
    season_types,
    entity_type
)
pd.DataFrame(stats)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,"(player_id, 203932)","(player_name, Aaron Gordon)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612743)","(team_abbreviation, DEN)","(games_played, 50)","(wins, 29)","(losses, 21)","(minutes, 1384.0)","(fgm, 51.0)","(fga, 158.0)","(points, 120.0)","(fg3m, 18.0)","(fg3a, 66.0)"
1,"(player_id, 1628988)","(player_name, Aaron Holiday)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612754)","(team_abbreviation, IND)","(games_played, 66)","(wins, 30)","(losses, 36)","(minutes, 1176.0)","(fgm, 40.0)","(fga, 107.0)","(points, 94.0)","(fg3m, 14.0)","(fg3a, 46.0)"
2,"(player_id, 1630174)","(player_name, Aaron Nesmith)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612738)","(team_abbreviation, BOS)","(games_played, 46)","(wins, 22)","(losses, 24)","(minutes, 669.0)","(fgm, 11.0)","(fga, 31.0)","(points, 25.0)","(fg3m, 3.0)","(fg3a, 15.0)"
3,"(player_id, 1627846)","(player_name, Abdel Nader)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612756)","(team_abbreviation, PHX)","(games_played, 24)","(wins, 16)","(losses, 8)","(minutes, 355.0)","(fgm, 2.0)","(fga, 6.0)","(points, 4.0)","(fg3m, 0)","(fg3a, 2.0)"
4,"(player_id, 1629690)","(player_name, Adam Mokoka)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612741)","(team_abbreviation, CHI)","(games_played, 14)","(wins, 3)","(losses, 11)","(minutes, 56.0)","(fgm, 0)","(fga, 2.0)","(points, 0)","(fg3m, 0)","(fg3a, 2.0)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,"(player_id, 1627812)","(player_name, Yogi Ferrell)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612746)","(team_abbreviation, LAC)","(games_played, 10)","(wins, 4)","(losses, 6)","(minutes, 137.0)","(fgm, 9.0)","(fga, 31.0)","(points, 21.0)","(fg3m, 3.0)","(fg3a, 15.0)"
536,"(player_id, 1629139)","(player_name, Yuta Watanabe)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612761)","(team_abbreviation, TOR)","(games_played, 50)","(wins, 24)","(losses, 26)","(minutes, 723.0)","(fgm, 7.0)","(fga, 28.0)","(points, 16.0)","(fg3m, 2.0)","(fg3a, 5.0)"
537,"(player_id, 203897)","(player_name, Zach LaVine)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612741)","(team_abbreviation, CHI)","(games_played, 58)","(wins, 26)","(losses, 32)","(minutes, 2034.0)","(fgm, 209.0)","(fga, 496.0)","(points, 541.0)","(fg3m, 123.0)","(fg3a, 315.0)"
538,"(player_id, 1630192)","(player_name, Zeke Nnaji)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612743)","(team_abbreviation, DEN)","(games_played, 42)","(wins, 30)","(losses, 12)","(minutes, 397.0)","(fgm, 2.0)","(fga, 5.0)","(points, 5.0)","(fg3m, 1.0)","(fg3a, 1.0)"


In [ ]:
#preview statistics related to speed and distance traveled for each player
stat_measure = 'SpeedDistance'
seasons = ['2020-21']
season_types = ['Regular Season']
entity_type = 'Player'
stats, league_totals = tracking.aggregate_full_season_tracking_stats_for_seasons(
    stat_measure,
    seasons,
    season_types,
    entity_type
)
pd.DataFrame(stats)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,"(player_id, 203932)","(player_name, Aaron Gordon)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612743)","(team_abbreviation, DEN)","(games_played, 50)","(wins, 29)","(losses, 21)","(minutes, 1384.0)","(dist_feet, 523828.0)","(dist_miles, 99.2)","(dist_miles_off, 54.9)","(dist_miles_def, 44.4)","(avg_speed, 4.06)","(avg_speed_off, 4.4)","(avg_speed_def, 3.71)"
1,"(player_id, 1628988)","(player_name, Aaron Holiday)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612754)","(team_abbreviation, IND)","(games_played, 66)","(wins, 30)","(losses, 36)","(minutes, 1176.0)","(dist_feet, 479903.0)","(dist_miles, 90.9)","(dist_miles_off, 47.0)","(dist_miles_def, 43.9)","(avg_speed, 4.32)","(avg_speed_off, 4.7)","(avg_speed_def, 3.98)"
2,"(player_id, 1630174)","(player_name, Aaron Nesmith)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612738)","(team_abbreviation, BOS)","(games_played, 46)","(wins, 22)","(losses, 24)","(minutes, 669.0)","(dist_feet, 274976.0)","(dist_miles, 52.1)","(dist_miles_off, 27.2)","(dist_miles_def, 24.9)","(avg_speed, 4.34)","(avg_speed_off, 4.53)","(avg_speed_def, 4.15)"
3,"(player_id, 1627846)","(player_name, Abdel Nader)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612756)","(team_abbreviation, PHX)","(games_played, 24)","(wins, 16)","(losses, 8)","(minutes, 355.0)","(dist_feet, 144913.0)","(dist_miles, 27.4)","(dist_miles_off, 14.8)","(dist_miles_def, 12.7)","(avg_speed, 4.41)","(avg_speed_off, 4.75)","(avg_speed_def, 4.1)"
4,"(player_id, 1629690)","(player_name, Adam Mokoka)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612741)","(team_abbreviation, CHI)","(games_played, 14)","(wins, 3)","(losses, 11)","(minutes, 56.0)","(dist_feet, 22367.0)","(dist_miles, 4.2)","(dist_miles_off, 2.2)","(dist_miles_def, 2.1)","(avg_speed, 4.13)","(avg_speed_off, 4.48)","(avg_speed_def, 3.99)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,"(player_id, 1627812)","(player_name, Yogi Ferrell)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612746)","(team_abbreviation, LAC)","(games_played, 10)","(wins, 4)","(losses, 6)","(minutes, 137.0)","(dist_feet, 47508.0)","(dist_miles, 9.0)","(dist_miles_off, 4.9)","(dist_miles_def, 4.1)","(avg_speed, 4.15)","(avg_speed_off, 4.43)","(avg_speed_def, 3.86)"
536,"(player_id, 1629139)","(player_name, Yuta Watanabe)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612761)","(team_abbreviation, TOR)","(games_played, 50)","(wins, 24)","(losses, 26)","(minutes, 723.0)","(dist_feet, 320658.0)","(dist_miles, 60.7)","(dist_miles_off, 33.6)","(dist_miles_def, 27.1)","(avg_speed, 4.68)","(avg_speed_off, 5.18)","(avg_speed_def, 4.19)"
537,"(player_id, 203897)","(player_name, Zach LaVine)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612741)","(team_abbreviation, CHI)","(games_played, 58)","(wins, 26)","(losses, 32)","(minutes, 2034.0)","(dist_feet, 830909.0)","(dist_miles, 157.4)","(dist_miles_off, 86.7)","(dist_miles_def, 70.7)","(avg_speed, 4.37)","(avg_speed_off, 4.81)","(avg_speed_def, 3.93)"
538,"(player_id, 1630192)","(player_name, Zeke Nnaji)","(team_name, None)","(season, 2020-21 Regular Season)","(game_id, None)","(opponent_team_id, None)","(team_id, 1610612743)","(team_abbreviation, DEN)","(games_played, 42)","(wins, 30)","(losses, 12)","(minutes, 397.0)","(dist_feet, 163977.0)","(dist_miles, 31.1)","(dist_miles_off, 16.8)","(dist_miles_def, 14.3)","(avg_speed, 4.

In [ ]:
def convert_to_dataframe(stats_list):
    dataframe_list = []
    for entry in stats_list:
        data_dict = dict(entry)  # Convert each entry to a dictionary
        dataframe_list.append(data_dict)
    return pd.DataFrame(dataframe_list)

In [ ]:
def create_tracking_dataframes(season_type, year, entity):

    """This function uses the NBA Tracking API to collect various tracking data and combines the data
    into a single dataframe. This function collects information from the following tracking statistics:

    -Drives
    -Catch and Shoot
    -Possessions
    -Pull Up Shots
    -Speed and Distance

    Documentation on the API can be found at:

    https://nba-stats-tracking.readthedocs.io/en/latest/source/quickstart.html#

    season_type: Can be Regular Season or Playoffs. Needs to be a string.

    year: The year for which tracking data will be collected. Should be a string formatted
    like XXXX-XX (i.e. 2017-18).

    entity_type: Whether you would like to look at team or player statistics. Should be a string."""

    stats_to_track = ['Drives', 'CatchShoot', 'Possessions', 'PullUpShot', 'SpeedDistance']

    tracking_stats_list = []

    for stat in stats_to_track:
        measurable, league_stats = tracking.aggregate_full_season_tracking_stats_for_seasons(
            stat, #type of tracking data to collect
            [year], #specific season
            [season_type], #regular season stats will be collected
            entity #player statistics will be collected
        )

        tracking_stats_list.append(measurable)
        time.sleep(3)

    #Create a dataframe for each type of stat
    drives_df = convert_to_dataframe(tracking_stats_list[0])       #Drives to the basket
    catch_shoot_df = convert_to_dataframe(tracking_stats_list[1])  # Catch and shoot
    poss_df = convert_to_dataframe(tracking_stats_list[2])         # Number of touches
    pullup_df = convert_to_dataframe(tracking_stats_list[3])       # Pull Up Shots
    speed_dist_df = convert_to_dataframe(tracking_stats_list[4])   # Speed and Distance Metrics

    #Create a new dataframe combining pertinent columns from each type of tracking statistic
    tracking_combined_df = pd.DataFrame([])

    #select columns from drives_df

    if entity_type == 'Player':
        tracking_combined_df['PLAYER_ID'] = drives_df['player_id']
        tracking_combined_df['PLAYER_NAME'] = drives_df['player_name']

    tracking_combined_df['TEAM_ID'] = drives_df['team_id']
    tracking_combined_df['TEAM_ABBR'] = drives_df['team_abbreviation']
    tracking_combined_df['DRIVES'] = drives_df['drives']
    tracking_combined_df['DRIVE_FGM'] = drives_df['fgm']
    tracking_combined_df['DRIVE_FGA'] = drives_df['fga']
    tracking_combined_df['DRIVE_PASSES'] = drives_df['passes']

    # Select columns from catch_shoot_df
    tracking_combined_df['CATCH_SHOOT_FGM'] = catch_shoot_df['fgm']
    tracking_combined_df['CATCH_SHOOT_FGA'] = catch_shoot_df['fga']
    tracking_combined_df['CATCH_SHOOT_FG3M'] = catch_shoot_df['fg3m']
    tracking_combined_df['CATCH_SHOOT_FG3A'] = catch_shoot_df['fg3a']

    # Select columns from poss_df
    tracking_combined_df['TOUCHES'] = poss_df['touches']
    tracking_combined_df['FRONT_CT_TOUCHES'] = poss_df['front_court_touches']
    tracking_combined_df['TIME_OF_POSS'] = poss_df['time_of_poss']

    # Select columns from pullup_df
    tracking_combined_df['PULL_UP_FGM'] = pullup_df['fgm']
    tracking_combined_df['PULL_UP_FGA'] = pullup_df['fga']
    tracking_combined_df['PULL_UP_FG3M'] = pullup_df['fg3m']
    tracking_combined_df['PULL_UP_FG3A'] = pullup_df['fg3a']

    # Select columns from speed_dist_df
    tracking_combined_df['DIST_MILES'] = speed_dist_df['dist_miles']
    tracking_combined_df['DIST_MILES_OFF'] = speed_dist_df['dist_miles_off']
    tracking_combined_df['DIST_MILES_DEF'] = speed_dist_df['dist_miles_def']

    return tracking_combined_df

In [ ]:
nbatrack_20_21 = create_tracking_dataframes('Regular Season', '2020-21', 'Player')
nbatrack_20_21

,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBR,DRIVES,DRIVE_FGM,DRIVE_FGA,DRIVE_PASSES,CATCH_SHOOT_FGM,CATCH_SHOOT_FGA,...,TOUCHES,FRONT_CT_TOUCHES,TIME_OF_POSS,PULL_UP_FGM,PULL_UP_FGA,PULL_UP_FG3M,PULL_UP_FG3A,DIST_MILES,DIST_MILES_OFF,DIST_MILES_DEF
0,203932,Aaron Gordon,1610612743,DEN,228.0,39.0,90.0,83.0,48.0,123.0,...,2415.0,1291.0,145.8,51.0,158.0,18.0,66.0,99.2,54.9,44.4
1,1628988,Aaron Holiday,1610612754,IND,383.0,60.0,167.0,160.0,53.0,137.0,...,1620.0,1020.0,80.1,40.0,107.0,14.0,46.0,90.9,47.0,43.9
2,1630174,Aaron Nesmith,1610612738,BOS,73.0,12.0,28.0,30.0,37.0,93.0,...,767.0,435.0,21.4,11.0,31.0,3.0,15.0,52.1,27.2,24.9
3,1627846,Abdel Nader,1610612756,PHX,83.0,20.0,38.0,26.0,18.0,42.0,...,389.0,261.0,12.4,2.0,6.0,0.0,2.0,27.4,14.8,12.7
4,1629690,Adam Mokoka,1610612741,CHI,5.0,1.0,3.0,1.0,1.0,8.0,...,58.0,43.0,1.3,0.0,2.0,0.0,2.0,4.2,2.2,2.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,1627812,Yogi Ferrell,1610612746,LAC,28.0,4.0,11.0,13.0,7.0,14.0,...,246.0,91.0,22.3,9.0,31.0,3.0,15.0,9.0,4.9,4.1
536,1629139,Yuta Watanabe,1610612761,TOR,78.0,10.0,29.0,38.0,36.0,87.0,...,953.0,541.0,26.5,7.0,28.0,2.0,5.0,60.7,33.6,27.1
537,203897,Zach LaVine,1610612741,CHI,697.0,184.0,345.0,192.0,77.0,165.0,...,3946.0,1966.0,314.9,209.0,496.0,123.0,315.0,157.4,86.7,70.7
538,1630192,Zeke Nnaji,1610612743,DEN,8.0,0.0,4.0,2.0,26.0,64.0,...,406.0,245.0,10.1,2.0,5.0,1.0,1.0,31.1,16.8,14.3


In [ ]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.4 MB/s eta 0:00:00


In [ ]:
import unidecode

In [ ]:
def clean_names(name):
    """This function removes extraneous player IDs from player names collected from Basketball Reference.
       The function removes punctuation from names as name punction is not consistent across different
       sources. The function also removes suffixes from names as suffixes are not consistent across different
       sources.

       The function takes a variable 'name' which should be a string."""

    #Name from Basketball Reference will follow the logic in the first if statement

    if '\\' in name:
        name_new = name.split('\\')[0]
        if '.' or '*' in name_new:
            name_stripped = name_new.replace('.', '').replace('*', '') #remove punctuation
        else:
            name_stripped = name_new

    #Names from stats.nba.com will follow the logic in the unnested else statement

    else:
        name_new = name
        if '.' in name_new:
            name_stripped = name_new.replace('.', '') #remove punctuation
        else:
            name_stripped = name_new



    name_clean = unidecode.unidecode(name_stripped) #remove accents and special characters

    name_split = name_clean.split() #split into first name, last name, and suffix if applicable

    if len(name_split) < 2:
        name_final = name_split[0]

    else:
        name_final = name_split[0] + ' ' + name_split[1] #store only first and last names in variable

    return name_final

In [ ]:
def join_nba_sources(dataframe1, dataframe2, year):
    """This function joins dataframes created from Basketball Reference and stats.nba.com so each player has
    box score, advanced, and tracking statistics.

    dataframe1: Dataframe from Basketball Reference.

    dataframe2: Dataframe from stats.nba.com

    year: The season that corresponds with the two dataframes. Should be a string.

    Both dataframes should be the same length (i.e. contain the same listing of players.)"""

    df_bball_ref = dataframe1
    df_nba_track = dataframe2

    df_bball_ref.drop_duplicates(subset=['Player'], keep='first', inplace=True)
    df_nba_track.drop_duplicates(subset=['PLAYER_ID'], keep='first', inplace=True)

    df_nba_track.drop(columns=['PLAYER_ID', 'TEAM_ID', 'TEAM_ABBR'], inplace=True)

    df_bball_ref.rename(columns={'Player':'PLAYER_NAME'}, inplace=True)

    df_bball_ref.set_index('PLAYER_NAME', inplace=True)
    df_nba_track.set_index('PLAYER_NAME', inplace=True)

    if len(dataframe1) != len(dataframe2):
        return print('Error: Dataframes not the same length.')
    else:
        df_final = df_bball_ref.join(df_nba_track, how='left')
        df_final.reset_index(inplace=True)
        df_final['PLAYER_NAME'] = df_final['PLAYER_NAME'].map(lambda x: x + ' ' + f'({year})' )
        return df_final

In [ ]:
bballref_2021_df

,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,...,WS,WS/48,OBPM,DBPM,BPM,VORP,pct_of_FGA_twos,pct_of_FGA_threes,two_pt_fg_pct,three_pt_fg_pct
0,Julius Randle,26.0,NYK,PF,71.0,71.0,2667.0,11.3,24.8,0.456,...,7.8,0.140,2.9,0.9,3.8,3.9,0.706,0.294,0.474,0.411
1,RJ Barrett,20.0,NYK,SG,72.0,72.0,2511.0,9.3,21.1,0.441,...,4.1,0.078,-0.9,-0.3,-1.2,0.5,0.708,0.292,0.457,0.401
2,Nikola Jokić,25.0,DEN,C,72.0,72.0,2488.0,14.5,25.7,0.566,...,15.6,0.301,9.1,3.0,12.1,8.8,0.817,0.183,0.606,0.388
3,Buddy Hield,28.0,SAC,SG,71.0,71.0,2433.0,8.0,19.6,0.406,...,2.2,0.044,1.0,-1.6,-0.6,0.9,0.273,0.727,0.446,0.391
4,Damian Lillard,30.0,POR,PG,67.0,67.0,2398.0,12.2,27.1,0.451,...,10.4,0.209,7.5,-1.3,6.3,5.0,0.472,0.528,0.519,0.391
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
700,Iman Shumpert,30.0,BRK,SG,2.0,0.0,11.0,4.4,17.5,0.250,...,0.0,-0.212,-10.4,-2.2,-12.6,0.0,0.250,0.750,1.000,0.000
701,Anžejs Pasečņiks,25.0,WAS,C,1.0,0.0,6.0,0.0,7.7,0.000,...,-0.1,-1.113,-40.7,-5.9,-46.6,-0.1,0.000,1.000,NaN,0.000
702,Ashton Hagans,21.0,MIN,PG,2.0,0.0,4.0,0.0,0.0,NaN,...,0.0,-0.353,-13.7,-7.4,-21.1,0.0,NaN,NaN,NaN,NaN
703,Udonis Haslem,40.0,MIA,C,1.0,0.0,3.0,33.1,33.1,1.000,...,0.0,0.475,24.1,7.0,31.1,0.0,1.000,0.000,1.000,NaN


In [ ]:
nbatrack_20_21

,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBR,DRIVES,DRIVE_FGM,DRIVE_FGA,DRIVE_PASSES,CATCH_SHOOT_FGM,CATCH_SHOOT_FGA,...,TOUCHES,FRONT_CT_TOUCHES,TIME_OF_POSS,PULL_UP_FGM,PULL_UP_FGA,PULL_UP_FG3M,PULL_UP_FG3A,DIST_MILES,DIST_MILES_OFF,DIST_MILES_DEF
0,203932,Aaron Gordon,1610612743,DEN,228.0,39.0,90.0,83.0,48.0,123.0,...,2415.0,1291.0,145.8,51.0,158.0,18.0,66.0,99.2,54.9,44.4
1,1628988,Aaron Holiday,1610612754,IND,383.0,60.0,167.0,160.0,53.0,137.0,...,1620.0,1020.0,80.1,40.0,107.0,14.0,46.0,90.9,47.0,43.9
2,1630174,Aaron Nesmith,1610612738,BOS,73.0,12.0,28.0,30.0,37.0,93.0,...,767.0,435.0,21.4,11.0,31.0,3.0,15.0,52.1,27.2,24.9
3,1627846,Abdel Nader,1610612756,PHX,83.0,20.0,38.0,26.0,18.0,42.0,...,389.0,261.0,12.4,2.0,6.0,0.0,2.0,27.4,14.8,12.7
4,1629690,Adam Mokoka,1610612741,CHI,5.0,1.0,3.0,1.0,1.0,8.0,...,58.0,43.0,1.3,0.0,2.0,0.0,2.0,4.2,2.2,2.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,1627812,Yogi Ferrell,1610612746,LAC,28.0,4.0,11.0,13.0,7.0,14.0,...,246.0,91.0,22.3,9.0,31.0,3.0,15.0,9.0,4.9,4.1
536,1629139,Yuta Watanabe,1610612761,TOR,78.0,10.0,29.0,38.0,36.0,87.0,...,953.0,541.0,26.5,7.0,28.0,2.0,5.0,60.7,33.6,27.1
537,203897,Zach LaVine,1610612741,CHI,697.0,184.0,345.0,192.0,77.0,165.0,...,3946.0,1966.0,314.9,209.0,496.0,123.0,315.0,157.4,86.7,70.7
538,1630192,Zeke Nnaji,1610612743,DEN,8.0,0.0,4.0,2.0,26.0,64.0,...,406.0,245.0,10.1,2.0,5.0,1.0,1.0,31.1,16.8,14.3


In [ ]:
nba_df_2021 = join_nba_sources(bballref_2021_df, nbatrack_20_21, '20-21')

In [ ]:
nba_df_2021.head()

,PLAYER_NAME,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,...,TOUCHES,FRONT_CT_TOUCHES,TIME_OF_POSS,PULL_UP_FGM,PULL_UP_FGA,PULL_UP_FG3M,PULL_UP_FG3A,DIST_MILES,DIST_MILES_OFF,DIST_MILES_DEF
0,Julius Randle (20-21),26.0,NYK,PF,71.0,71.0,2667.0,11.3,24.8,0.456,...,5936.0,3434.0,332.8,238.0,573.0,44.0,110.0,188.0,105.2,82.8
1,RJ Barrett (20-21),20.0,NYK,SG,72.0,72.0,2511.0,9.3,21.1,0.441,...,3414.0,2490.0,193.4,64.0,192.0,6.0,20.0,192.3,104.3,88.1
2,Nikola Jokić (20-21),25.0,DEN,C,72.0,72.0,2488.0,14.5,25.7,0.566,...,7269.0,4173.0,333.0,71.0,152.0,6.0,16.0,169.7,92.8,76.9
3,Buddy Hield (20-21),28.0,SAC,SG,71.0,71.0,2433.0,8.0,19.6,0.406,...,3872.0,2298.0,175.8,159.0,403.0,104.0,271.0,182.2,97.0,85.2
4,Damian Lillard (20-21),30.0,POR,PG,67.0,67.0,2398.0,12.2,27.1,0.451,...,5449.0,1752.0,559.2,280.0,728.0,201.0,542.0,164.9,93.6,71.2


In [ ]:
nbatrack_21_22 = create_tracking_dataframes('Regular Season', '2021-22', 'Player')

df4 = pd.read_csv('nba_2021-22_player_per100pos.csv')
df5 = pd.read_csv('nba_2021-22_advanced.csv')
df6 = pd.read_csv('nba_2021-22_player_shooting.csv', header = [0, 1])

bballref_2122_df = join_dataframes(df4, df5, df6)

nba_df_2122 = join_nba_sources(bballref_2122_df, nbatrack_21_22, '21-22')

In [ ]:
nba_df_2122.head()

,PLAYER_NAME,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,...,TOUCHES,FRONT_CT_TOUCHES,TIME_OF_POSS,PULL_UP_FGM,PULL_UP_FGA,PULL_UP_FG3M,PULL_UP_FG3A,DIST_MILES,DIST_MILES_OFF,DIST_MILES_DEF
0,Mikal Bridges (21-22),25.0,PHO,SF,82.0,82.0,2854.0,7.7,14.5,0.534,...,2942.0,1859.0,85.8,98.0,207.0,9.0,29.0,212.5,108.9,103.5
1,Miles Bridges (21-22),23.0,CHO,PF,80.0,80.0,2837.0,10.1,20.5,0.491,...,4758.0,2998.0,187.0,71.0,240.0,45.0,161.0,210.7,114.5,96.3
2,DeMar DeRozan (21-22),32.0,CHI,PF,76.0,76.0,2743.0,13.8,27.3,0.504,...,4670.0,2568.0,417.7,458.0,929.0,7.0,23.0,189.2,105.0,84.2
3,Jayson Tatum (21-22),23.0,BOS,SF,76.0,76.0,2731.0,12.9,28.5,0.453,...,5533.0,3181.0,348.8,220.0,625.0,128.0,383.0,181.7,98.8,82.9
4,Saddiq Bey (21-22),22.0,DET,SF,82.0,82.0,2704.0,8.1,20.5,0.396,...,4052.0,2910.0,156.2,98.0,312.0,38.0,150.0,192.6,102.8,89.8


In [ ]:
nbatrack_22_23 = create_tracking_dataframes('Regular Season', '2022-23', 'Player')

df7 = pd.read_csv('nba_2022-23_player_per100pos.csv')
df8 = pd.read_csv('nba_2022-23_advanced.csv')
df9 = pd.read_csv('nba_2022-23_player_shooting.csv', header = [0, 1])

bballref_2223_df = join_dataframes(df7, df8, df9)

nba_df_2223 = join_nba_sources(bballref_2223_df, nbatrack_22_23, '22-23')

In [ ]:
nba_df_2223.head()

,PLAYER_NAME,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,...,TOUCHES,FRONT_CT_TOUCHES,TIME_OF_POSS,PULL_UP_FGM,PULL_UP_FGA,PULL_UP_FG3M,PULL_UP_FG3A,DIST_MILES,DIST_MILES_OFF,DIST_MILES_DEF
0,Mikal Bridges (22-23),26.0,2TM,SG,83.0,83.0,2963.0,9.8,20.9,0.468,...,4259.0,2618.0,192.9,222.0,489.0,28.0,78.0,224.0,120.9,103.1
1,Anthony Edwards (22-23),21.0,MIN,SG,79.0,79.0,2842.0,11.8,25.8,0.459,...,5568.0,2894.0,370.2,220.0,627.0,122.0,361.0,199.3,106.1,93.2
2,Zach LaVine (22-23),27.0,CHI,SG,77.0,77.0,2768.0,11.8,24.4,0.485,...,4708.0,2461.0,332.1,228.0,589.0,102.0,298.0,200.0,107.2,92.8
3,Nikola Vučević (22-23),32.0,CHI,C,82.0,82.0,2746.0,10.6,20.4,0.520,...,5374.0,3654.0,131.0,24.0,78.0,0.0,0.0,185.1,96.1,89.0
4,Julius Randle (22-23),28.0,NYK,PF,77.0,77.0,2737.0,11.9,25.9,0.459,...,5344.0,3350.0,267.3,201.0,512.0,78.0,233.0,191.9,106.2,85.8


In [ ]:
nbatrack_23_24 = create_tracking_dataframes('Regular Season', '2023-24', 'Player')

df10 = pd.read_csv('nba_2023-24_player_per100pos.csv')
df11 = pd.read_csv('nba_2023-24_advanced.csv')
df12 = pd.read_csv('nba_2023-24_player_shooting.csv', header = [0, 1])

bballref_2324_df = join_dataframes(df10, df11, df12)

nba_df_2324 = join_nba_sources(bballref_2324_df, nbatrack_23_24, '23-24')

In [ ]:
nba_df_2324.head()

,PLAYER_NAME,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,...,TOUCHES,FRONT_CT_TOUCHES,TIME_OF_POSS,PULL_UP_FGM,PULL_UP_FGA,PULL_UP_FG3M,PULL_UP_FG3A,DIST_MILES,DIST_MILES_OFF,DIST_MILES_DEF
0,DeMar DeRozan (23-24),34.0,CHI,SF,79.0,79.0,2989.0,10.8,22.6,0.480,...,5025.0,2658.0,395.9,291.0,643.0,14.0,47.0,200.9,109.9,91.0
1,Domantas Sabonis (23-24),27.0,SAC,C,82.0,82.0,2928.0,10.5,17.7,0.594,...,7502.0,4451.0,306.0,22.0,47.0,4.0,6.0,219.3,115.2,104.1
2,Coby White (23-24),23.0,CHI,PG,79.0,78.0,2881.0,9.4,20.9,0.447,...,5719.0,2404.0,371.3,130.0,321.0,59.0,146.0,203.8,108.8,95.1
3,Mikal Bridges (23-24),27.0,BRK,SF,82.0,82.0,2854.0,9.8,22.5,0.436,...,4556.0,2915.0,222.2,163.0,423.0,43.0,127.0,211.7,117.3,94.5
4,Paolo Banchero (23-24),21.0,ORL,PF,80.0,80.0,2799.0,11.3,24.9,0.455,...,6113.0,2981.0,439.5,231.0,604.0,66.0,205.0,201.8,113.4,88.4


In [ ]:
nbatrack_24_25 = create_tracking_dataframes('Regular Season', '2024-25', 'Player')

df13 = pd.read_csv('nba_2024-25_player_per100pos.csv')
df14 = pd.read_csv('nba_2024-25_advanced.csv')
df15 = pd.read_csv('nba_2024-25_player_shooting.csv', header=[0, 1])

bballref_2425_df = join_dataframes(df13, df14, df15)

nba_df_2425 = join_nba_sources(bballref_2425_df, nbatrack_24_25, '24-25')

In [ ]:
nba_df_2425.head()

,PLAYER_NAME,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,...,TOUCHES,FRONT_CT_TOUCHES,TIME_OF_POSS,PULL_UP_FGM,PULL_UP_FGA,PULL_UP_FG3M,PULL_UP_FG3A,DIST_MILES,DIST_MILES_OFF,DIST_MILES_DEF
0,Mikal Bridges (24-25),28.0,NYK,SF,82.0,82.0,3036.0,9.7,19.3,0.500,...,4456.0,3106.0,185.2,168.0,376.0,15.0,58.0,231.1,126.0,105.0
1,Josh Hart (24-25),29.0,NYK,SG,77.0,77.0,2897.0,6.9,13.2,0.525,...,5391.0,2369.0,251.3,58.0,151.0,24.0,72.0,211.0,116.1,94.9
2,Anthony Edwards (24-25),23.0,MIN,SG,79.0,79.0,2871.0,12.4,27.7,0.447,...,5836.0,2967.0,465.2,333.0,881.0,230.0,597.0,201.1,111.2,90.0
3,Devin Booker (24-25),28.0,PHO,SG,75.0,75.0,2795.0,11.6,25.1,0.461,...,5433.0,3626.0,368.4,329.0,758.0,93.0,303.0,198.9,111.6,87.2
4,James Harden (24-25),35.0,LAC,PG,79.0,79.0,2789.0,9.4,22.9,0.410,...,6470.0,2513.0,588.3,260.0,739.0,184.0,545.0,175.3,95.6,79.7


In [ ]:
nba_df_2021.to_csv('nba_df_2021.csv')


In [ ]:
nba_df_2122.to_csv('nba_df_2122.csv')

In [ ]:
nba_df_2223.to_csv('nba_df_2223.csv')

In [ ]:
nba_df_2324.to_csv('nba_df_2324.csv')

In [ ]:
nba_df_2425.to_csv('nba_df_2425.csv')

In [ ]:
nba_df_20_25 = pd.concat([nba_df_2021, nba_df_2122, nba_df_2223, nba_df_2324, nba_df_2425], ignore_index=True)
nba_df_20_25.head()

,PLAYER_NAME,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,...,TOUCHES,FRONT_CT_TOUCHES,TIME_OF_POSS,PULL_UP_FGM,PULL_UP_FGA,PULL_UP_FG3M,PULL_UP_FG3A,DIST_MILES,DIST_MILES_OFF,DIST_MILES_DEF
0,Julius Randle (20-21),26.0,NYK,PF,71.0,71.0,2667.0,11.3,24.8,0.456,...,5936.0,3434.0,332.8,238.0,573.0,44.0,110.0,188.0,105.2,82.8
1,RJ Barrett (20-21),20.0,NYK,SG,72.0,72.0,2511.0,9.3,21.1,0.441,...,3414.0,2490.0,193.4,64.0,192.0,6.0,20.0,192.3,104.3,88.1
2,Nikola Jokić (20-21),25.0,DEN,C,72.0,72.0,2488.0,14.5,25.7,0.566,...,7269.0,4173.0,333.0,71.0,152.0,6.0,16.0,169.7,92.8,76.9
3,Buddy Hield (20-21),28.0,SAC,SG,71.0,71.0,2433.0,8.0,19.6,0.406,...,3872.0,2298.0,175.8,159.0,403.0,104.0,271.0,182.2,97.0,85.2
4,Damian Lillard (20-21),30.0,POR,PG,67.0,67.0,2398.0,12.2,27.1,0.451,...,5449.0,1752.0,559.2,280.0,728.0,201.0,542.0,164.9,93.6,71.2


In [ ]:
nba_df_20_25.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2825 entries, 0 to 2824
Data columns (total 73 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   PLAYER_NAME        2825 non-null   object 
 1   Age                2825 non-null   float64
 2   Team               2825 non-null   object 
 3   Pos                2825 non-null   object 
 4   G                  2825 non-null   float64
 5   GS                 2825 non-null   float64
 6   MP                 2825 non-null   float64
 7   FG                 2825 non-null   float64
 8   FGA                2825 non-null   float64
 9   FG%                2806 non-null   float64
 10  3P                 2825 non-null   float64
 11  3PA                2825 non-null   float64
 12  3P%                2686 non-null   float64
 13  2P                 2825 non-null   float64
 14  2PA                2825 non-null   float64
 15  2P%                2786 non-null   float64
 16  eFG%               2806 

In [ ]:
nba_df_20_25.to_csv('nba_df_20_25.csv')